# Derin Evrişimli Sinir Ağları - Kapsamlı Rehber

Bu notebook aşağıdaki mimarileri kapsamaktadır:
- **7.1** AlexNet
- **7.2** VGG (Blokları Kullanan Ağlar)
- **7.3** NiN (Network in Network)
- **7.4** GoogLeNet (Inception)
- **7.5** Toplu Normalleştirme (Batch Normalization)

Framework: **TensorFlow / Keras**

## Gerekli Kütüphaneler

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
import time

print(f"TensorFlow sürümü: {tf.__version__}")
print(f"GPU kullanılabilir mi: {len(tf.config.list_physical_devices('GPU')) > 0}")

## Yardımcı Fonksiyonlar

In [ ]:
def load_fashion_mnist(batch_size, resize=None):
    """Fashion-MNIST veri kümesini yükler ve isteğe bağlı olarak yeniden boyutlandırır."""
    (x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
    
    # Normalize et
    x_train = x_train.astype('float32') / 255.0
    x_test  = x_test.astype('float32') / 255.0
    
    # Kanal boyutu ekle
    x_train = x_train[..., np.newaxis]
    x_test  = x_test[..., np.newaxis]
    
    # Yeniden boyutlandır
    if resize:
        x_train = tf.image.resize(x_train, [resize, resize])
        x_test  = tf.image.resize(x_test,  [resize, resize])
    
    train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))\
                              .shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_ds  = tf.data.Dataset.from_tensor_slices((x_test, y_test))\
                              .batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return train_ds, test_ds


def train_model(model, train_ds, test_ds, num_epochs, lr, model_name="Model"):
    """Modeli derler, eğitir ve sonuçları görselleştirir."""
    model.compile(
        optimizer=keras.optimizers.SGD(learning_rate=lr),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )
    
    start = time.time()
    history = model.fit(
        train_ds,
        epochs=num_epochs,
        validation_data=test_ds,
        verbose=1
    )
    elapsed = time.time() - start
    
    # Sonuçları görselleştir
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['loss'],     label='Eğitim Kaybı')
    axes[0].plot(history.history['val_loss'], label='Test Kaybı')
    axes[0].set_title(f'{model_name} - Kayıp')
    axes[0].set_xlabel('Dönem'); axes[0].set_ylabel('Kayıp')
    axes[0].legend()

    axes[1].plot(history.history['accuracy'],     label='Eğitim Doğruluğu')
    axes[1].plot(history.history['val_accuracy'], label='Test Doğruluğu')
    axes[1].set_title(f'{model_name} - Doğruluk')
    axes[1].set_xlabel('Dönem'); axes[1].set_ylabel('Doğruluk')
    axes[1].legend()

    plt.tight_layout()
    plt.show()
    
    final_train_acc = history.history['accuracy'][-1]
    final_test_acc  = history.history['val_accuracy'][-1]
    print(f"\n{'='*50}")
    print(f"{model_name} Sonuçları:")
    print(f"  Eğitim Doğruluğu : {final_train_acc:.4f}")
    print(f"  Test Doğruluğu   : {final_test_acc:.4f}")
    print(f"  Toplam Süre      : {elapsed:.1f} saniye")
    print(f"{'='*50}")
    
    return history


def print_layer_shapes(model, input_shape):
    """Her katmanın çıktı şeklini yazdırır."""
    x = tf.random.uniform(input_shape)
    print(f"Girdi şekli: {x.shape}")
    print("-" * 45)
    for layer in model.layers:
        x = layer(x)
        print(f"{layer.__class__.__name__:<25} çıktı: {x.shape}")

---
## 7.1 AlexNet

AlexNet, 2012 ImageNet yarışmasını kazanan ve derin öğrenme çağını başlatan 8 katmanlı CNN'dir.

**LeNet'e göre temel farklar:**
- Sigmoid yerine **ReLU** aktivasyon fonksiyonu
- Aşırı öğrenmeyi önlemek için **Dropout**
- Çok daha büyük filtre sayıları ve derin mimari
- Veri zenginleştirme teknikleri

In [ ]:
def build_alexnet(num_classes=10):
    """AlexNet mimarisi (Fashion-MNIST için uyarlanmış)."""
    return keras.Sequential([
        # 1. Evrişim: 11x11 çekirdek, 4'lük uzun adım
        layers.Conv2D(96, kernel_size=11, strides=4, activation='relu',
                      input_shape=(224, 224, 1)),
        layers.MaxPooling2D(pool_size=3, strides=2),

        # 2. Evrişim: 5x5 çekirdek, dolgu ile boyutu koru
        layers.Conv2D(256, kernel_size=5, padding='same', activation='relu'),
        layers.MaxPooling2D(pool_size=3, strides=2),

        # 3-4-5. Evrişim: 3x3 çekirdek, ardışık
        layers.Conv2D(384, kernel_size=3, padding='same', activation='relu'),
        layers.Conv2D(384, kernel_size=3, padding='same', activation='relu'),
        layers.Conv2D(256, kernel_size=3, padding='same', activation='relu'),
        layers.MaxPooling2D(pool_size=3, strides=2),

        # Tam bağlı katmanlar
        layers.Flatten(),
        layers.Dense(4096, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(4096, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes)  # Çıktı katmanı
    ], name='AlexNet')


alexnet = build_alexnet()
alexnet.summary()

In [ ]:
# Katman çıktı şekillerini incele
print("AlexNet Katman Çıktı Şekilleri:")
print("-" * 45)
print_layer_shapes(alexnet, (1, 224, 224, 1))

In [ ]:
# AlexNet'i eğit
batch_size = 128
train_iter, test_iter = load_fashion_mnist(batch_size, resize=224)

alexnet_history = train_model(
    model=alexnet,
    train_ds=train_iter,
    test_ds=test_iter,
    num_epochs=10,
    lr=0.01,
    model_name="AlexNet"
)

---
## 7.2 VGG - Blokları Kullanan Ağlar

VGG, Oxford Üniversitesi'nin Görsel Geometri Grubu tarafından geliştirilmiştir. **Tekrar eden bloklar** kullanarak ağ tasarımını modüler hale getirir.

**Temel fikir:** Birden fazla 3×3 evrişimden oluşan bloklar + MaxPooling

In [ ]:
def vgg_block(num_convs, num_channels):
    """Bir VGG bloğu: num_convs adet 3x3 evrişim + MaxPooling."""
    block = keras.Sequential()
    for _ in range(num_convs):
        block.add(layers.Conv2D(num_channels, kernel_size=3,
                                padding='same', activation='relu'))
    block.add(layers.MaxPooling2D(pool_size=2, strides=2))
    return block


def build_vgg(conv_arch, num_classes=10):
    """
    VGG ağı.
    conv_arch: [(num_convs, num_channels), ...] formatında blok tanımları
    """
    net = keras.Sequential(name='VGG')
    
    # İlk katman için girdi şeklini belirt
    first = True
    for i, (num_convs, num_channels) in enumerate(conv_arch):
        block = keras.Sequential(name=f'vgg_block_{i+1}')
        for j in range(num_convs):
            if first:
                block.add(layers.Conv2D(num_channels, kernel_size=3,
                                        padding='same', activation='relu',
                                        input_shape=(224, 224, 1)))
                first = False
            else:
                block.add(layers.Conv2D(num_channels, kernel_size=3,
                                        padding='same', activation='relu'))
        block.add(layers.MaxPooling2D(pool_size=2, strides=2))
        net.add(block)
    
    # Tam bağlı katmanlar
    net.add(layers.Flatten())
    net.add(layers.Dense(4096, activation='relu'))
    net.add(layers.Dropout(0.5))
    net.add(layers.Dense(4096, activation='relu'))
    net.add(layers.Dropout(0.5))
    net.add(layers.Dense(num_classes))
    
    return net


# VGG-11 mimarisi (orijinal)
conv_arch_vgg11 = ((1, 64), (1, 128), (2, 256), (2, 512), (2, 512))

# Fashion-MNIST için küçültülmüş versiyon (kanal sayısı 4'e bölünmüş)
ratio = 4
small_conv_arch = [(n, c // ratio) for n, c in conv_arch_vgg11]
print("Küçük VGG mimarisi:", small_conv_arch)

vgg_net = build_vgg(small_conv_arch)
vgg_net.summary()

In [ ]:
# VGG'yi eğit
train_iter, test_iter = load_fashion_mnist(batch_size=128, resize=224)

vgg_history = train_model(
    model=vgg_net,
    train_ds=train_iter,
    test_ds=test_iter,
    num_epochs=10,
    lr=0.05,
    model_name="VGG (Küçük)"
)

---
## 7.3 NiN - Network in Network

NiN, her piksel konumuna tam bağlı katman uygulayan **1×1 evrişimler** kullanır.

**Temel farklar:**
- Tam bağlı katmanlar **yok** — parametre sayısı çok az
- Onların yerine **Global Average Pooling**
- Her NiN bloğu: 1 büyük evrişim + 2 adet 1×1 evrişim

In [ ]:
def nin_block(num_channels, kernel_size, strides, padding):
    """NiN bloğu: 1 evrişim + 2 adet 1x1 evrişim."""
    block = keras.Sequential([
        layers.Conv2D(num_channels, kernel_size=kernel_size,
                      strides=strides, padding=padding, activation='relu'),
        layers.Conv2D(num_channels, kernel_size=1, activation='relu'),
        layers.Conv2D(num_channels, kernel_size=1, activation='relu')
    ])
    return block


def build_nin(num_classes=10):
    """NiN modeli."""
    net = keras.Sequential(name='NiN')
    
    # 1. NiN Bloğu
    net.add(layers.InputLayer(input_shape=(224, 224, 1)))
    net.add(nin_block(96,  kernel_size=11, strides=4, padding='valid'))
    net.add(layers.MaxPooling2D(pool_size=3, strides=2))
    
    # 2. NiN Bloğu
    net.add(nin_block(256, kernel_size=5,  strides=1, padding='same'))
    net.add(layers.MaxPooling2D(pool_size=3, strides=2))
    
    # 3. NiN Bloğu
    net.add(nin_block(384, kernel_size=3,  strides=1, padding='same'))
    net.add(layers.MaxPooling2D(pool_size=3, strides=2))
    net.add(layers.Dropout(0.5))
    
    # Çıktı bloğu: kanal sayısını sınıf sayısına indir
    net.add(nin_block(num_classes, kernel_size=3, strides=1, padding='same'))
    
    # Global Average Pooling — tam bağlı katman yok!
    net.add(layers.GlobalAveragePooling2D())
    
    return net


nin_net = build_nin()
nin_net.summary()

In [ ]:
# NiN'i eğit
train_iter, test_iter = load_fashion_mnist(batch_size=128, resize=224)

nin_history = train_model(
    model=nin_net,
    train_ds=train_iter,
    test_ds=test_iter,
    num_epochs=10,
    lr=0.1,
    model_name="NiN"
)

---
## 7.4 GoogLeNet - Inception

GoogLeNet, farklı boyutlardaki filtreleri **paralel** olarak kullanan Inception bloklarından oluşur.

**Inception bloğu:** 4 paralel yol:
1. 1×1 evrişim
2. 1×1 → 3×3 evrişim
3. 1×1 → 5×5 evrişim
4. 3×3 MaxPool → 1×1 evrişim

In [ ]:
class InceptionBlock(layers.Layer):
    """GoogLeNet Inception Bloğu — 4 paralel yol."""
    
    def __init__(self, c1, c2, c3, c4, **kwargs):
        """
        c1: Yol 1 çıktı kanalı (1x1 evrişim)
        c2: (Yol 2 bottleneck, Yol 2 çıktı) — 1x1 -> 3x3
        c3: (Yol 3 bottleneck, Yol 3 çıktı) — 1x1 -> 5x5
        c4: Yol 4 çıktı kanalı — MaxPool -> 1x1
        """
        super().__init__(**kwargs)
        
        # Yol 1: 1×1 evrişim
        self.p1 = layers.Conv2D(c1, kernel_size=1, activation='relu', padding='same')
        
        # Yol 2: 1×1 → 3×3
        self.p2_1 = layers.Conv2D(c2[0], kernel_size=1, activation='relu', padding='same')
        self.p2_2 = layers.Conv2D(c2[1], kernel_size=3, activation='relu', padding='same')
        
        # Yol 3: 1×1 → 5×5
        self.p3_1 = layers.Conv2D(c3[0], kernel_size=1, activation='relu', padding='same')
        self.p3_2 = layers.Conv2D(c3[1], kernel_size=5, activation='relu', padding='same')
        
        # Yol 4: MaxPool → 1×1
        self.p4_1 = layers.MaxPooling2D(pool_size=3, strides=1, padding='same')
        self.p4_2 = layers.Conv2D(c4,    kernel_size=1, activation='relu', padding='same')
    
    def call(self, x):
        p1 = self.p1(x)
        p2 = self.p2_2(self.p2_1(x))
        p3 = self.p3_2(self.p3_1(x))
        p4 = self.p4_2(self.p4_1(x))
        # Kanal boyutunda birleştir
        return tf.concat([p1, p2, p3, p4], axis=-1)


def build_googlenet(num_classes=10):
    """GoogLeNet (Inception v1) — 96x96 girdi için uyarlanmış."""
    inputs = keras.Input(shape=(96, 96, 1))
    
    # Modül 1
    x = layers.Conv2D(64, kernel_size=7, strides=2, padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D(pool_size=3, strides=2, padding='same')(x)
    
    # Modül 2
    x = layers.Conv2D(64,  kernel_size=1, activation='relu')(x)
    x = layers.Conv2D(192, kernel_size=3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D(pool_size=3, strides=2, padding='same')(x)
    
    # Modül 3: 2 Inception bloğu
    x = InceptionBlock(64,  (96,  128), (16, 32),  32,  name='inception_3a')(x)
    x = InceptionBlock(128, (128, 192), (32, 96),  64,  name='inception_3b')(x)
    x = layers.MaxPooling2D(pool_size=3, strides=2, padding='same')(x)
    
    # Modül 4: 5 Inception bloğu
    x = InceptionBlock(192, (96,  208), (16, 48),  64,  name='inception_4a')(x)
    x = InceptionBlock(160, (112, 224), (24, 64),  64,  name='inception_4b')(x)
    x = InceptionBlock(128, (128, 256), (24, 64),  64,  name='inception_4c')(x)
    x = InceptionBlock(112, (144, 288), (32, 64),  64,  name='inception_4d')(x)
    x = InceptionBlock(256, (160, 320), (32, 128), 128, name='inception_4e')(x)
    x = layers.MaxPooling2D(pool_size=3, strides=2, padding='same')(x)
    
    # Modül 5: 2 Inception bloğu
    x = InceptionBlock(256, (160, 320), (32, 128), 128, name='inception_5a')(x)
    x = InceptionBlock(384, (192, 384), (48, 128), 128, name='inception_5b')(x)
    
    # Global Average Pooling + çıktı
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes)(x)
    
    return keras.Model(inputs, outputs, name='GoogLeNet')


googlenet = build_googlenet()
googlenet.summary()

In [ ]:
# GoogLeNet'i eğit (96x96 girdi)
train_iter, test_iter = load_fashion_mnist(batch_size=128, resize=96)

googlenet_history = train_model(
    model=googlenet,
    train_ds=train_iter,
    test_ds=test_iter,
    num_epochs=10,
    lr=0.1,
    model_name="GoogLeNet"
)

---
## 7.5 Toplu Normalleştirme (Batch Normalization)

Toplu normalleştirme, her mini-grup için ara katman çıktılarını normalize eder. Bu sayede:
- Eğitim **hızlanır**
- Daha yüksek **öğrenme oranları** kullanılabilir
- **Düzenlileştirici** etki sağlar

### 7.5.1 Sıfırdan Batch Normalization Uygulaması

In [ ]:
class BatchNormManual(layers.Layer):
    """Sıfırdan yazılmış Batch Normalization katmanı."""
    
    def __init__(self, num_features, num_dims, momentum=0.9, eps=1e-5, **kwargs):
        """
        num_features: Tam bağlı için çıktı sayısı, evrişimli için kanal sayısı
        num_dims    : Tam bağlı için 2, evrişimli için 4
        """
        super().__init__(**kwargs)
        self.num_dims  = num_dims
        self.momentum  = momentum
        self.eps       = eps
        
        shape = (1, num_features) if num_dims == 2 else (1, 1, 1, num_features)
        
        # Öğrenilebilir ölçek (gamma) ve kaydırma (beta) parametreleri
        self.gamma = self.add_weight(name='gamma', shape=shape,
                                     initializer='ones',  trainable=True)
        self.beta  = self.add_weight(name='beta',  shape=shape,
                                     initializer='zeros', trainable=True)
        
        # Hareketli ortalama ve varyans (öğrenilmez, takip edilir)
        self.moving_mean = self.add_weight(name='moving_mean', shape=shape,
                                           initializer='zeros', trainable=False)
        self.moving_var  = self.add_weight(name='moving_var',  shape=shape,
                                           initializer='ones',  trainable=False)
    
    def call(self, X, training=None):
        if not training:
            # Tahmin modu: hareketli ortalama ve varyans kullan
            X_hat = (X - self.moving_mean) / tf.sqrt(self.moving_var + self.eps)
        else:
            # Eğitim modu: mini-grup istatistikleri hesapla
            if self.num_dims == 2:
                # Tam bağlı katman: öznitelik boyutunda
                mean = tf.reduce_mean(X, axis=0, keepdims=True)
                var  = tf.reduce_mean((X - mean)**2, axis=0, keepdims=True)
            else:
                # Evrişimli katman: uzamsal boyutlarda (H, W)
                mean = tf.reduce_mean(X, axis=(0, 1, 2), keepdims=True)
                var  = tf.reduce_mean((X - mean)**2, axis=(0, 1, 2), keepdims=True)
            
            X_hat = (X - mean) / tf.sqrt(var + self.eps)
            
            # Hareketli istatistikleri güncelle
            self.moving_mean.assign(self.momentum * self.moving_mean + (1 - self.momentum) * mean)
            self.moving_var.assign( self.momentum * self.moving_var  + (1 - self.momentum) * var)
        
        # Ölçek ve kaydır
        return self.gamma * X_hat + self.beta

In [ ]:
def build_lenet_with_bn(use_custom_bn=True, num_classes=10):
    """
    LeNet + Batch Normalization.
    use_custom_bn=True  -> elle yazılmış BatchNorm
    use_custom_bn=False -> Keras BatchNormalization
    """
    BN = (lambda nf, nd: BatchNormManual(nf, nd)) if use_custom_bn \
         else (lambda nf, nd: layers.BatchNormalization())
    
    return keras.Sequential([
        layers.InputLayer(input_shape=(28, 28, 1)),
        
        # 1. Evrişimli blok
        layers.Conv2D(6, kernel_size=5),
        BN(6, 4),
        layers.Activation('sigmoid'),
        layers.AveragePooling2D(pool_size=2, strides=2),
        
        # 2. Evrişimli blok
        layers.Conv2D(16, kernel_size=5),
        BN(16, 4),
        layers.Activation('sigmoid'),
        layers.AveragePooling2D(pool_size=2, strides=2),
        
        # Tam bağlı katmanlar
        layers.Flatten(),
        layers.Dense(120),
        BN(120, 2),
        layers.Activation('sigmoid'),
        
        layers.Dense(84),
        BN(84, 2),
        layers.Activation('sigmoid'),
        
        layers.Dense(num_classes)
    ], name='LeNet_BN')


# Özel BatchNorm ile LeNet
lenet_bn_custom = build_lenet_with_bn(use_custom_bn=True)
lenet_bn_custom.summary()

In [ ]:
# LeNet + Özel BN'i eğit (28x28 — yeniden boyutlandırma yok)
train_iter, test_iter = load_fashion_mnist(batch_size=256)

lenet_bn_history = train_model(
    model=lenet_bn_custom,
    train_ds=train_iter,
    test_ds=test_iter,
    num_epochs=10,
    lr=1.0,
    model_name="LeNet + Özel BN"
)

In [ ]:
# Keras yerleşik BatchNormalization ile karşılaştır
lenet_bn_keras = build_lenet_with_bn(use_custom_bn=False)

lenet_bn_keras_history = train_model(
    model=lenet_bn_keras,
    train_ds=train_iter,
    test_ds=test_iter,
    num_epochs=10,
    lr=1.0,
    model_name="LeNet + Keras BN"
)

In [ ]:
# Öğrenilen gamma ve beta parametrelerini incele
print("İlk BN katmanının öğrenilen parametreleri (Keras BN):")
for layer in lenet_bn_keras.layers:
    if isinstance(layer, layers.BatchNormalization):
        print(f"  gamma: {layer.gamma.numpy().flatten()}")
        print(f"  beta : {layer.beta.numpy().flatten()}")
        break

---
## Mimarilerin Karşılaştırması

In [ ]:
def count_params(model):
    return sum([tf.size(w).numpy() for w in model.trainable_weights])

models_info = {
    'AlexNet'       : alexnet,
    'VGG (Küçük)'   : vgg_net,
    'NiN'           : nin_net,
    'GoogLeNet'     : googlenet,
    'LeNet+BN'      : lenet_bn_keras,
}

print(f"{'Model':<18} {'Parametre Sayısı':>20}")
print("-" * 40)
for name, model in models_info.items():
    n = count_params(model)
    print(f"{name:<18} {n:>20,}")

In [ ]:
# Tüm modellerin son test doğruluklarını karşılaştır
results = {
    'AlexNet'     : alexnet_history.history['val_accuracy'][-1],
    'VGG (Küçük)' : vgg_history.history['val_accuracy'][-1],
    'NiN'         : nin_history.history['val_accuracy'][-1],
    'GoogLeNet'   : googlenet_history.history['val_accuracy'][-1],
    'LeNet+BN'    : lenet_bn_history.history['val_accuracy'][-1],
}

names = list(results.keys())
accs  = [results[n] * 100 for n in names]

plt.figure(figsize=(10, 5))
bars = plt.bar(names, accs, color=['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2'])
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')
plt.ylim(0, 105)
plt.ylabel('Test Doğruluğu (%)')
plt.title('Fashion-MNIST — CNN Mimarilerinin Karşılaştırması')
plt.tight_layout()
plt.show()

---
## Özet

| Model | Yenilik | Öne Çıkan Özellik |
|-------|---------|-------------------|
| **AlexNet** | ReLU, Dropout, GPU eğitimi | Derin CNN'lerin kapısını açtı |
| **VGG** | Tekrar eden bloklar | 3×3 evrişimlerle derinlik |
| **NiN** | 1×1 evrişim, Global Avg Pool | Parametre verimliliği |
| **GoogLeNet** | Inception bloğu | Paralel çok ölçekli özellik |
| **Batch Norm** | Mini-grup normalleştirme | Hızlı ve kararlı eğitim |